In [1]:
import time
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "sentence-transformers/multi-qa-mpnet-base-cos-v1"

In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

text = "Nas będą z tego głównie interesowały „przepustowości łączy”, czyli ile bitów danych jesteśmy w stanie przesłać przez sieć komputerową w czasie jednej sekundy"
inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt")

In [3]:
### 1. Pure PyTorch time

times_pure_torch = []

for _ in range(100):
    start_time = time.time()
    model(**inputs)
    end_time = time.time()
    times_pure_torch.append(end_time - start_time)

print("Avg: ", sum(times_pure_torch) / len(times_pure_torch))

Avg:  0.08096377849578858


In [4]:
model.eval()

MPNetModel(
  (embeddings): MPNetEmbeddings(
    (word_embeddings): Embedding(30527, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): MPNetEncoder(
    (layer): ModuleList(
      (0-11): 12 x MPNetLayer(
        (attention): MPNetAttention(
          (attn): MPNetSelfAttention(
            (q): Linear(in_features=768, out_features=768, bias=True)
            (k): Linear(in_features=768, out_features=768, bias=True)
            (v): Linear(in_features=768, out_features=768, bias=True)
            (o): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (intermediate): MPNetIntermediate(
          (dense): Linear(in_

In [5]:
### model.eval() time
times_eval = []
for _ in range(100):
    start_time = time.time()
    with torch.no_grad():
        model(**inputs)
    end_time = time.time()
    times_eval.append(end_time - start_time)

print("Avg eval: ", sum(times_eval) / len(times_eval))

Avg eval:  0.08721730709075928


In [6]:
### no_grad_eval() time

times_no_grad_eval = []
for _ in range(100):
    start_time = time.time()
    with torch.no_grad():
        model(**inputs)
    end_time = time.time()
    times_no_grad_eval.append(end_time - start_time)

print("Avg no_grad eval: ", sum(times_no_grad_eval) / len(times_no_grad_eval))

Avg no_grad eval:  0.08938513040542602


In [7]:
### inference_mode() time

times_inference_mode = []
for _ in range(100):
    start_time = time.time()
    with torch.inference_mode():
        model(**inputs)
    end_time = time.time()
    times_inference_mode.append(end_time - start_time)

print("Avg inference_mode: ", sum(times_inference_mode) / len(times_inference_mode))

Avg inference_mode:  0.08911252021789551


In [9]:
avg_pure = sum(times_pure_torch) / len(times_pure_torch)
avg_eval = sum(times_eval) / len(times_eval)
avg_no_grad = sum(times_no_grad_eval) / len(times_no_grad_eval)
avg_inference = sum(times_inference_mode) / len(times_inference_mode)

eval_multiplier = avg_eval / avg_pure
no_grad_multiplier = avg_no_grad / avg_pure
inference_multiplier = avg_inference / avg_pure

print(f"eval is {eval_multiplier:.2f}x of pure")
print(f"no_grad is {no_grad_multiplier:.2f}x of pure")
print(f"inference_mode is {inference_multiplier:.2f}x of pure")

eval is 1.08x of pure
no_grad is 1.10x of pure
inference_mode is 1.10x of pure


In [10]:
## Torch compile

compiled_model = torch.compile(model)

#warm-up
warm_start = time.time()
with torch.inference_mode():
    compiled_model(**inputs)
warm_end = time.time()
print("Warm-up time: ", warm_end - warm_start)

### compiled model time

times_compiled = []

for _ in range(100):
    start_time = time.time()
    with torch.inference_mode():
        compiled_model(**inputs)
    end_time = time.time()
    times_compiled.append(end_time - start_time)

print("Avg compiled inference_mode: ", sum(times_compiled) / len(times_compiled))

Warm-up time:  18.55934190750122
Avg compiled inference_mode:  0.0652764654159546


In [11]:
print("Compiled is ", (sum(times_compiled) / len(times_compiled)) / avg_inference, "x of inference_mode")
print("Compiled is ", (sum(times_compiled) / len(times_compiled)) / avg_pure, "x of pure")
print("Compiled is ", (sum(times_compiled) / len(times_compiled)) / avg_eval, "x of eval")
print("Compiled is ", (sum(times_compiled) / len(times_compiled)) / avg_no_grad, "x of no_grad")

Compiled is  0.7325173304081443 x of inference_mode
Compiled is  0.8062428239975241 x of pure
Compiled is  0.7484347727914505 x of eval
Compiled is  0.7302832710527887 x of no_grad


In [12]:
from torch.ao.quantization import quantize_dynamic

quantized_model = quantize_dynamic(
    model,
    {torch.nn.Linear},
    dtype=torch.qint8
)

### quantized model time

times_quantized = []

for _ in range(100):
    start_time = time.time()
    with torch.inference_mode():
        quantized_model(**inputs)
    end_time = time.time()
    times_quantized.append(end_time - start_time)
print("Avg quantized inference_mode: ", sum(times_quantized) / len(times_quantized))

print("Quantized is ", (sum(times_quantized) / len(times_quantized)) / avg_inference, "x of inference_mode")
print("Quantized is ", (sum(times_quantized) / len(times_quantized)) /
        avg_pure, "x of pure")
print("Quantized is ", (sum(times_quantized) / len(times_quantized)) /
        avg_eval, "x of eval")
print("Quantized is ", (sum(times_quantized) / len(times_quantized)) /
        avg_no_grad, "x of no_grad")
print("Quantized is ", (sum(times_quantized) / len(times_quantized)) /
        (sum(times_compiled) / len(times_compiled)), "x of compiled")

Avg quantized inference_mode:  0.053471500873565676
Quantized is  0.6000447607453882 x of inference_mode
Quantized is  0.660437319836142 x of pure
Quantized is  0.6130836029817184 x of eval
Quantized is  0.5982147212968629 x of no_grad
Quantized is  0.8191543542199269 x of compiled


In [16]:
### compare size of models

import os
torch.save(model.state_dict(), "model.pth")
torch.save(quantized_model.state_dict(), "quantized_model.pth")
size_model = os.path.getsize("model.pth")
size_quantized_model = os.path.getsize("quantized_model.pth")

print("Original model size: ", size_model / (1024 * 1024), "MB")
print("Quantized model size: ", size_quantized_model / (1024 * 1024), "MB")

print("Size reduction: ", size_model / size_quantized_model, "x")

Original model size:  417.7161388397217 MB
Quantized model size:  173.09671783447266 MB
Size reduction:  2.4131950279910646 x
